# Irreversibility project: Kaggle runner

Settings: **Accelerator = GPU T4 x2**, **Internet = on**, secret **HF_TOKEN** added (Add-ons -> Secrets).
Attach the uploaded dataset `irrev-code`. Run cells top to bottom. **Stop after the pilot (cell 4) and send `pilot_results.zip` to Claude** before running the full cells.

**If a session ends (timeout, quota, closed tab):** start a new session, attach your latest `results_<tag>.zip` as an input, run cells 1, 1b, 2, then re-run the step that was cut off. It resumes where it stopped. If a step ended with a red error, re-running it starts that step from scratch (the failed output is kept in a `.failed-...` folder).

In [ ]:
# 1. setup: find the uploaded package wherever Kaggle put it (extracted folder or the zip itself)
import subprocess, os
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
src = sh("find /kaggle/input -maxdepth 6 -type d -name irrev_pkg | head -1")
if src:
    sh(f'rm -rf /kaggle/working/irrev_pkg && cp -r "{src}" /kaggle/working/irrev_pkg')
else:
    z = sh("find /kaggle/input -maxdepth 6 -name 'irrev_kaggle_package*.zip' | head -1")
    print('zip:', z)
    sh(f'cd /kaggle/working && rm -rf irrev_pkg && unzip -q "{z}"')
print('package ready:', os.path.exists('/kaggle/working/irrev_pkg/irrev/agent.py'))
%cd /kaggle/working/irrev_pkg
!pip install -q "transformers==5.22.1" accelerate
!python -c "import torch,transformers;print('torch',torch.__version__,'| transformers',transformers.__version__,'|',torch.cuda.device_count(),'GPUs')"

In [ ]:
# 1b. RESTORE results from an earlier session (safe to run every time; does nothing on the first session)
# Attach your earlier results as an input first: either re-upload results_<tag>.zip as a Kaggle Dataset,
# or Add Input -> your previous notebook version's output. Finished items are then skipped,
# interrupted runs resume, and runs that ended in an error start again from scratch.
import glob, os, subprocess
os.makedirs('results', exist_ok=True)
zips = glob.glob('/kaggle/input/**/results_*.zip', recursive=True)
dirs = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/results/*/*.status.json', recursive=True)]
for z in zips:
    subprocess.run(f'unzip -n -q "{z}" -d .', shell=True)      # -n: never overwrite newer local files
for d in sorted(set(dirs)):
    subprocess.run(f'cp -rn "{d}/results/." results/', shell=True)
st = glob.glob('results/**/*.status.json', recursive=True)
print('restored from:', zips + sorted(set(dirs)) or 'nothing (first session)')
for s in sorted(st):
    import json; print(' ', s.replace('.status.json',''), '->', json.load(open(s))['state'])

In [ ]:
# 2. Hugging Face login from the Kaggle secret
from kaggle_secrets import UserSecretsClient
import os
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
!hf auth login --token $HF_TOKEN 2>/dev/null || huggingface-cli login --token $HF_TOKEN

In [ ]:
# 3. GPU check (paste the output to Claude)
MODEL = 'Qwen/Qwen2.5-7B-Instruct'   # later: 'meta-llama/Llama-3.1-8B-Instruct'
TAG = 'qwen7b'                        # later: 'llama8b'
!python gpu_check.py $MODEL

In [ ]:
# 4. PILOT: 20 probe items, 16 stated items, 8 dev agent episodes (about 10-20 min)
!python -m irrev.extract --model $MODEL --data data/probe_sandbox.jsonl --out pilot/$TAG/hidden --limit 20 --shard 20
!python -m irrev.stated  --model $MODEL --data data/probe_sandbox.jsonl --out pilot/$TAG/stated.jsonl --limit 16 --batch 8
!python -m irrev.agent   --model $MODEL --tasks data/tasks.jsonl --split dev --out pilot/$TAG/agent_dev.jsonl --limit 8 --batch 8
!zip -qr /kaggle/working/pilot_results.zip pilot
print('Download /kaggle/working/pilot_results.zip (Output panel) and send it to Claude.')

## Full runs (only after Claude has checked the pilot)
Each command resumes where it stopped if the session ends. After each one, zip and download `results`.

In [ ]:
# 5. RQ1 hidden states (sandbox, then ScienceWorld when Claude sends it)
!python -m irrev.extract --model $MODEL --data data/probe_sandbox.jsonl --out results/$TAG/hidden_sandbox
!zip -qr /kaggle/working/results_$TAG.zip results   # download this after each step

In [ ]:
# 6. RQ2 stated answers (probe items + task shortcuts for the blindness gap)
!python -m irrev.stated --model $MODEL --data data/probe_sandbox.jsonl --out results/$TAG/stated_sandbox.jsonl --batch 8
!python -m irrev.stated --model $MODEL --data data/tasks_shortcut_stated.jsonl --out results/$TAG/stated_tasks.jsonl --batch 8
!zip -qr /kaggle/working/results_$TAG.zip results   # download this after each step

In [ ]:
# 7. RQ3 agent runs: neutral prompt, then the careful-prompt ablation
!python -m irrev.agent --model $MODEL --tasks data/tasks.jsonl --split test --out results/$TAG/agent_neutral.jsonl --batch 8
!python -m irrev.agent --model $MODEL --tasks data/tasks.jsonl --split test --out results/$TAG/agent_careful.jsonl --batch 8 --careful
!zip -qr /kaggle/working/results_$TAG.zip results   # download this after each step

In [ ]:
# 8. package results for Claude
!zip -qr /kaggle/working/results_$TAG.zip results
print('Download /kaggle/working/results_'+TAG+'.zip')